In [ ]:
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
import numpy as np
import pandas as pd


In [ ]:
# External validation demo (replace with real NSCLC-Radiomics predictions)
from src.evaluation.external_validation import compute_external_metrics

# Simulate external predictions
np.random.seed(99)
y_true_ext = np.random.randint(0, 2, 80)
y_prob_ext = np.clip(np.random.rand(80), 0.05, 0.95)

# Internal AUC from Phase 2 (load from saved results in practice)
internal_auc = 0.93

ext_df = compute_external_metrics(
    y_true_ext, y_prob_ext,
    internal_auc=internal_auc,
    output_csv='/content/external_validation.csv',
)
print(ext_df[['auc', 'auc_ci_lo', 'auc_ci_hi', 'delta_auc_internal_to_external']].to_string())


In [ ]:
# Route B ICC comparison (synthetic demo)
from src.evaluation.external_validation import compute_route_b_icc_delta_auc
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

n_feat = 15
feat_names = [f'original_glcm_f{i}' for i in range(8)] +              [f'original_shape_f{i}' for i in range(7)]
labels_df_ext = pd.DataFrame({'label': np.random.randint(0, 2, 80)})

feat_a = pd.DataFrame(np.random.randn(80, n_feat), columns=feat_names)
feat_b = feat_a + np.random.randn(80, n_feat) * 0.1  # small perturbation

scaler = StandardScaler().fit(feat_a.values)
X_train = scaler.transform(feat_a.values)
clf = xgb.XGBClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, labels_df_ext['label'].values)

icc_summary = compute_route_b_icc_delta_auc(
    feat_a, feat_b, labels_df_ext, clf, scaler, feat_names,
    output_csv='/content/route_b_icc.csv',
)
print(icc_summary)
